# ABSA Perfume Reviews - Inference Pipeline

**Goal:** load the two fine-tuned models from `best_model/`, run them on a
fresh batch of reviews (e.g. newly scraped Tokopedia reviews), and export a
tidy CSV with aspect + sentiment + confidence per row - ready for the
analysis notebook.

This notebook does **not** retrain anything. If `best_model/` doesn't exist
yet, run the training notebook first.


# 0. Setup

## 0.1 Install & Imports
**Objective:** minimal dependencies - no `datasets`/`Trainer` needed here,
just plain `transformers` inference.

In [7]:
!pip install -q transformers torch pandas tqdm

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option('display.max_colwidth', None)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0.2 Configuration
**Objective:** point at your saved model folder and the new review file to
score.

In [13]:
BASE_DIR = "/content/drive/MyDrive/PORTFOLIO #4 - ABSA Perfume"

SAVE_DIR = f"{BASE_DIR}/output/best_models"
NEW_REVIEWS_PATH = f"{BASE_DIR}/data/raw/perfume_review_raw.csv"
OUTPUT_PATH = f"{BASE_DIR}/output/absa_predictions.csv"
MAX_LEN = 256

ASPECT_THRESHOLD = 0.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


# 1. Load Models

## 1.1 Load Aspect & Sentiment Models
**Objective:** load both fine-tuned models plus the label mapping saved
during training, straight from disk (this is what a production script would
do - no dependency on the training notebook's in-memory objects).

In [16]:
aspect_tok = AutoTokenizer.from_pretrained(os.path.join(SAVE_DIR, "aspect_model"))
aspect_mdl = AutoModelForSequenceClassification.from_pretrained(
    os.path.join(SAVE_DIR, "aspect_model")
).to(DEVICE).eval()

sent_tok = AutoTokenizer.from_pretrained(os.path.join(SAVE_DIR, "sentiment_model"))
sent_mdl = AutoModelForSequenceClassification.from_pretrained(
    os.path.join(SAVE_DIR, "sentiment_model")
).to(DEVICE).eval()

with open(os.path.join(SAVE_DIR, "label_mappings.json")) as f:
    mappings = json.load(f)

ASPECT_LABELS = mappings["aspect_labels"]
ID2SENT = {int(k): v for k, v in mappings["id2sent"].items()}
NO_ASPECT_LABEL = mappings["no_aspect_label"]

print("Aspect labels:", ASPECT_LABELS)
print("Sentiment labels:", ID2SENT)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Aspect labels: ['Aroma', 'Longevity', 'Packaging', 'Projection', 'Value for Money']
Sentiment labels: {0: 'negative', 1: 'neutral', 2: 'positive'}


# 2. Load New Reviews

## 2.1 Load Raw Review Data
**Objective:** load the scraped reviews (output of `run.py` /
`review_tokopedia.csv`) that haven't been through ABSA yet.

In [17]:
raw_df = pd.read_csv(NEW_REVIEWS_PATH)
raw_df = raw_df.dropna(subset=["review"]).reset_index(drop=True)

if "review_id" not in raw_df.columns:
    raw_df["review_id"] = [f"RVW_{i:05d}" for i in range(1, len(raw_df) + 1)]

print("Total reviews to score:", len(raw_df))
raw_df.head()

Total reviews to score: 5579


,brand,product,username,rating,tanggal,review,review_id
0,HMNS,Philea,J***a,5,3 bulan lalu,"Wanginya pas buat saya..elegant, ringan, seger..box kemasannya keren, botolnya juga simple elwgant, keren..gak nyangka aja ownernya alumni ITB..😄",RVW_00001
1,HMNS,Philea,Z***f,5,10 bulan lalu,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,RVW_00002
2,HMNS,Philea,F***m,5,7 bulan lalu,"produk sesuai deskripsi, produk berkualitas, harga bersaing, pengemasan baik, pengiriman baik",RVW_00003
3,HMNS,Philea,Q***z,5,3 minggu lalu,"pengiriman aman dan cepat, belanja dapet harga oke dan pastinya di jamin ori berawal dari nyobain bodymist nya dan suka banget aroma segarnya yang ringan dan nyari tau untuk produk parfum dan langsung pesan ternyata aroma nya beda tipis tipis aja untuk ketahanan belom cobain",RVW_00004
4,HMNS,Philea,shabrina,5,10 bulan lalu,"packing aman banget, wangi ny bener2 se enakk itu, apa lagi pas udh dry done nya🥹🥹🥹, the besttt pwwollllll",RVW_00005


# 3. Batch Inference

## 3.1 Batched Aspect Prediction
**Objective:** predict aspects for all reviews in batches (much faster than
one-by-one) - returns, per review, the list of detected aspects with
confidence.

In [18]:
def predict_aspects_batch(texts, batch_size=32, threshold=ASPECT_THRESHOLD):
    all_results = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Aspect detection"):
        batch = texts[i:i + batch_size]
        inputs = aspect_tok(
            batch, truncation=True, max_length=MAX_LEN,
            padding=True, return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            logits = aspect_mdl(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()

        for row_probs in probs:
            detected = [
                (ASPECT_LABELS[j], float(row_probs[j]))
                for j in range(len(ASPECT_LABELS)) if row_probs[j] >= threshold
            ]
            all_results.append(detected)

    return all_results

aspect_predictions = predict_aspects_batch(raw_df["review"].astype(str).tolist())
raw_df["detected_aspects"] = aspect_predictions
raw_df.head()

Aspect detection:   0%|          | 0/175 [00:00<?, ?it/s]

,brand,product,username,rating,tanggal,review,review_id,detected_aspects
0,HMNS,Philea,J***a,5,3 bulan lalu,"Wanginya pas buat saya..elegant, ringan, seger..box kemasannya keren, botolnya juga simple elwgant, keren..gak nyangka aja ownernya alumni ITB..😄",RVW_00001,"[(Aroma, 0.9782111644744873), (Packaging, 0.9966603517532349)]"
1,HMNS,Philea,Z***f,5,10 bulan lalu,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,RVW_00002,"[(Aroma, 0.9571763277053833), (Longevity, 0.9775872826576233), (Packaging, 0.9949907660484314)]"
2,HMNS,Philea,F***m,5,7 bulan lalu,"produk sesuai deskripsi, produk berkualitas, harga bersaing, pengemasan baik, pengiriman baik",RVW_00003,"[(Packaging, 0.9910240173339844), (Value for Money, 0.9837151169776917)]"
3,HMNS,Philea,Q***z,5,3 minggu lalu,"pengiriman aman dan cepat, belanja dapet harga oke dan pastinya di jamin ori berawal dari nyobain bodymist nya dan suka banget aroma segarnya yang ringan dan nyari tau untuk produk parfum dan langsung pesan ternyata aroma nya beda tipis tipis aja untuk ketahanan belom cobain",RVW_00004,"[(Aroma, 0.9533882737159729), (Value for Money, 0.99806147813797)]"
4,HMNS,Philea,shabrina,5,10 bulan lalu,"packing aman banget, wangi ny bener2 se enakk itu, apa lagi pas udh dry done nya🥹🥹🥹, the besttt pwwollllll",RVW_00005,"[(Aroma, 0.988857090473175), (Packaging, 0.9903952479362488)]"


## 3.2 Batched Sentiment Prediction (per detected aspect)
**Objective:** for every (review, aspect) pair detected in Step 3.1, predict
sentiment + confidence. Reviews with zero detected aspects contribute one
row with `aspect="No Aspect"` and no sentiment - so no data silently
disappears.

In [19]:
def build_sentiment_pairs(df):
    pairs = []
    for _, row in df.iterrows():
        if len(row["detected_aspects"]) == 0:
            pairs.append((row["review_id"], row["review"], NO_ASPECT_LABEL, None))
        else:
            for aspect, aspect_conf in row["detected_aspects"]:
                pairs.append((row["review_id"], row["review"], aspect, aspect_conf))
    return pairs

pairs = build_sentiment_pairs(raw_df)
print("Total (review, aspect) rows to score:", len(pairs))

Total (review, aspect) rows to score: 8099


In [20]:
def predict_sentiment_batch(pairs, batch_size=32):
    results = []
    real_pairs = [p for p in pairs if p[2] != NO_ASPECT_LABEL]
    no_aspect_pairs = [p for p in pairs if p[2] == NO_ASPECT_LABEL]

    for i in tqdm(range(0, len(real_pairs), batch_size), desc="Sentiment classification"):
        batch = real_pairs[i:i + batch_size]
        texts = [b[1] for b in batch]
        aspects = [b[2] for b in batch]

        inputs = sent_tok(
            texts, aspects, truncation=True, max_length=MAX_LEN,
            padding=True, return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            logits = sent_mdl(**inputs).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()

        for (review_id, text, aspect, aspect_conf), row_probs in zip(batch, probs):
            pred_id = int(np.argmax(row_probs))
            results.append({
                "review_id": review_id,
                "review_text": text,
                "aspect": aspect,
                "aspect_confidence": round(aspect_conf, 4) if aspect_conf is not None else None,
                "sentiment": ID2SENT[pred_id],
                "sentiment_confidence": round(float(row_probs[pred_id]), 4),
            })

    for review_id, text, aspect, _ in no_aspect_pairs:
        results.append({
            "review_id": review_id,
            "review_text": text,
            "aspect": aspect,
            "aspect_confidence": None,
            "sentiment": None,
            "sentiment_confidence": None,
        })

    return pd.DataFrame(results)

predictions_df = predict_sentiment_batch(pairs)
print("Total output rows:", len(predictions_df))
predictions_df.head(10)

Sentiment classification:   0%|          | 0/226 [00:00<?, ?it/s]

Total output rows: 8099


,review_id,review_text,aspect,aspect_confidence,sentiment,sentiment_confidence
0,RVW_00001,"Wanginya pas buat saya..elegant, ringan, seger..box kemasannya keren, botolnya juga simple elwgant, keren..gak nyangka aja ownernya alumni ITB..😄",Aroma,0.9782,positive,0.9998
1,RVW_00001,"Wanginya pas buat saya..elegant, ringan, seger..box kemasannya keren, botolnya juga simple elwgant, keren..gak nyangka aja ownernya alumni ITB..😄",Packaging,0.9967,positive,0.9998
2,RVW_00002,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,Aroma,0.9572,positive,0.9998
3,RVW_00002,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,Longevity,0.9776,positive,0.9998
4,RVW_00002,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,Packaging,0.9950,positive,0.9998
5,RVW_00003,"produk sesuai deskripsi, produk berkualitas, harga bersaing, pengemasan baik, pengiriman baik",Packaging,0.9910,positive,0.9998
6,RVW_00003,"produk sesuai deskripsi, produk berkualitas, harga bersaing, pengemasan baik, pengiriman baik",Value for Money,0.9837,positive,0.9998
7,RVW_00004,"pengiriman aman dan cepat, belanja dapet harga oke dan pastinya di jamin ori berawal dari nyobain bodymist nya dan suka banget aroma segarnya yang ringan dan nyari tau untuk produk parfum dan langsung pesan ternyata aroma nya beda tipis tipis aja untuk ketahanan belom cobain",Aroma,0.9534,positive,0.9998
8,RVW_00004,"pengiriman aman dan cepat, belanja dapet harga oke dan pastinya di jamin ori berawal dari nyobain bodymist nya dan suka banget aroma segarnya yang ringan dan nyari tau untuk produk parfum dan langsung pesan ternyata aroma nya beda tipis tipis aja untuk ketahanan belom cobain",Value for Money,0.9981,positive,0.9983
9,RVW_00005,"packing aman banget, wangi ny bener2 se enakk itu, apa lagi pas udh dry done nya🥹🥹🥹, the besttt pwwollllll",Aroma,0.9889,positive,0.9998


# 4. Merge Metadata & Export

## 4.1 Merge Back Brand/Product Metadata
**Objective:** bring `brand`, `product`, `rating`, `tanggal` back in from the
raw scrape so the analysis notebook can do brand/product comparisons without
re-joining anything.

In [21]:
meta_cols = [c for c in ["review_id", "brand", "product", "rating", "tanggal"] if c in raw_df.columns]
final_df = predictions_df.merge(raw_df[meta_cols], on="review_id", how="left")

print(final_df.shape)
final_df.head(10)

(8099, 10)


,review_id,review_text,aspect,aspect_confidence,sentiment,sentiment_confidence,brand,product,rating,tanggal
0,RVW_00001,"Wanginya pas buat saya..elegant, ringan, seger..box kemasannya keren, botolnya juga simple elwgant, keren..gak nyangka aja ownernya alumni ITB..😄",Aroma,0.9782,positive,0.9998,HMNS,Philea,5,3 bulan lalu
1,RVW_00001,"Wanginya pas buat saya..elegant, ringan, seger..box kemasannya keren, botolnya juga simple elwgant, keren..gak nyangka aja ownernya alumni ITB..😄",Packaging,0.9967,positive,0.9998,HMNS,Philea,5,3 bulan lalu
2,RVW_00002,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,Aroma,0.9572,positive,0.9998,HMNS,Philea,5,10 bulan lalu
3,RVW_00002,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,Longevity,0.9776,positive,0.9998,HMNS,Philea,5,10 bulan lalu
4,RVW_00002,Fix ini parfum hmns terbaik terenak terdebes. Bagi penyuka Citrus sih bakal SUKA BGTTTT. Tapi klo ga gitu suka wangi jejerukan bakal gasuka wkwk. Ketahanannya bisa 6 jam di baju duh cinta. Packagingnya juga gemes bgt. Makasih ya udh nyiptain iniii,Packaging,0.9950,positive,0.9998,HMNS,Philea,5,10 bulan lalu
5,RVW_00003,"produk sesuai deskripsi, produk berkualitas, harga bersaing, pengemasan baik, pengiriman baik",Packaging,0.9910,positive,0.9998,HMNS,Philea,5,7 bulan lalu
6,RVW_00003,"produk sesuai deskripsi, produk berkualitas, harga bersaing, pengemasan baik, pengiriman baik",Value for Money,0.9837,positive,0.9998,HMNS,Philea,5,7 bulan lalu
7,RVW_00004,"pengiriman aman dan cepat, belanja dapet harga oke dan pastinya di jamin ori berawal dari nyobain bodymist nya dan suka banget aroma segarnya yang ringan dan nyari tau untuk produk parfum dan langsung pesan ternyata aroma nya beda tipis tipis aja untuk ketahanan belom cobain",Aroma,0.9534,positive,0.9998,HMNS,Philea,5,3 minggu lalu
8,RVW_00004,"pengiriman aman dan cepat, belanja dapet harga oke dan pastinya di jamin ori berawal dari nyobain bodymist nya dan suka banget aroma segarnya yang ringan dan nyari tau untuk produk parfum dan langsung pesan ternyata aroma nya beda tipis tipis aja untuk ketahanan belom cobain",Value for Money,0.9981,positive,0.9983,HMNS,Philea,5,3 minggu lalu
9,RVW_00005,"packing aman banget, wangi ny bener2 se enakk itu, apa lagi pas udh dry done nya🥹🥹🥹, the besttt pwwollllll",Aroma,0.9889,positive,0.9998,HMNS,Philea,5,10 bulan lalu


## 4.2 Save to CSV
**Objective:** one clean, analysis-ready CSV: one row per (review, aspect),
with confidence scores, ready to feed into the analysis/dashboard notebook.

In [23]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved {len(final_df)} rows to {OUTPUT_PATH}")

Saved 8099 rows to /content/drive/MyDrive/PORTFOLIO #4 - ABSA Perfume/output/absa_predictions.csv


## 4.3 Quick Sanity Check
**Objective:** confirm nothing silently dropped - row counts should line up
with what we expect from Step 3.

In [22]:
print("Unique reviews scored:", final_df['review_id'].nunique())
print("Reviews with No Aspect:", (final_df['aspect'] == NO_ASPECT_LABEL).sum())
print("\nAspect distribution:")
print(final_df['aspect'].value_counts())
print("\nSentiment distribution:")
print(final_df['sentiment'].value_counts(dropna=False))

Unique reviews scored: 5579
Reviews with No Aspect: 897

Aspect distribution:
aspect
Aroma              4082
Packaging          1324
Longevity          1023
No Aspect           897
Value for Money     430
Projection          343
Name: count, dtype: int64

Sentiment distribution:
sentiment
positive    6600
None         897
negative     356
neutral      246
Name: count, dtype: int64
